# 🧱 B2B Sales Lakehouse & Analytics Pipeline
### Medallion-Architektur (Bronze ➔ Silver ➔ Gold) mit Databricks, PySpark & Delta Lake

Dieses Projekt demonstriert eine strukturierte Data-Engineering- und Analytics-Pipeline europäischer B2B-Auftragsdaten (Fokus: **DACH-Region** - Deutschland, Österreich, Schweiz) für das Executive Reporting.

---

### 🏛️ Architektur-Übersicht:
1. **Bronze-Layer (`raw_orders`)**: Ingestion der rohen Quelldatei `data/b2b_sales_raw.csv` mit explizitem Schema und Audit-Metadaten.
2. **Silver-Layer (`cleaned_orders`)**: Bereinigung, Deduplizierung, Data-Quality-Gates (Assertions) und Neuberechnung betriebswirtschaftlicher Kennzahlen.
3. **Gold-Layer (`executive_sales_gold`)**: Analytische Data Marts mit gewichteter Margenberechnung und Drilldown-Analysen (Region × Kundensegment) für das Tableau Executive Dashboard.

> ℹ️ **Hinweis zur Datenbasis:** 4.591 synthetisch generierte B2B-Transaktionen (Cloud-Lizenzen, Plattform & Beratung) im Zeitraum 01.01.2025 bis 25.10.2026.


## 1. Setup & Spark-Session Initialisierung


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# SparkSession für lokale Ausführung oder Databricks Runtime
try:
    spark
    DATABRICKS_ENV = True
except NameError:
    spark = SparkSession.builder \
        .appName("B2B_Sales_Medallion_Lakehouse") \
        .getOrCreate()
    DATABRICKS_ENV = False

DATA_PATH = "../data/b2b_sales_raw.csv"
print(f"🚀 Spark Session aktiv | Umgebung: {'Databricks Cluster' if DATABRICKS_ENV else 'Lokal / CI'}")


🚀 Spark Session aktiv | Umgebung: Databricks Cluster


## 2. Bronze Layer: Raw Data Ingestion
Explizites Schema zur Vermeidung von Typkonvertierungsfehlern (`Schema-on-Read`). Technische Metadaten für Datenherkunft (Data Lineage).


In [2]:
schema_bronze = T.StructType([
    T.StructField("Bestell_ID", T.StringType(), False),
    T.StructField("Bestelldatum", T.DateType(), True),
    T.StructField("Jahr", T.IntegerType(), True),
    T.StructField("Quartal", T.StringType(), True),
    T.StructField("Region", T.StringType(), True),
    T.StructField("Land", T.StringType(), True),
    T.StructField("Stadt", T.StringType(), True),
    T.StructField("Kundensegment", T.StringType(), True),
    T.StructField("Vertriebskanal", T.StringType(), True),
    T.StructField("Produktkategorie", T.StringType(), True),
    T.StructField("Produktname", T.StringType(), True),
    T.StructField("Menge", T.IntegerType(), True),
    T.StructField("Rabatt_Quote", T.DoubleType(), True),
    T.StructField("Umsatz_EUR", T.DoubleType(), True),
    T.StructField("Kosten_EUR", T.DoubleType(), True),
    T.StructField("Gewinn_EUR", T.DoubleType(), True),
    T.StructField("Gewinnmarge_Prozent", T.DoubleType(), True),
    T.StructField("Zielumsatz_EUR", T.DoubleType(), True),
])

df_bronze = (
    spark.read
    .option("header", "true")
    .option("encoding", "UTF-8")
    .schema(schema_bronze)
    .csv(DATA_PATH)
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("b2b_sales_raw.csv"))
)

total_bronze = df_bronze.count()
print(f"📥 Bronze-Layer erfolgreich geladen: {total_bronze:,} Datensätze")
df_bronze.select("Bestell_ID", "Bestelldatum", "Region", "Land", "Umsatz_EUR", "Gewinn_EUR").show(5, truncate=False)


📥 Bronze-Layer erfolgreich geladen: 4,591 Datensätze
+----------+------------+----------+------------+----------+----------+
|Bestell_ID|Bestelldatum|Region    |Land        |Umsatz_EUR|Gewinn_EUR|
+----------+------------+----------+------------+----------+----------+
|AUF-10001 |2025-01-01  |DACH      |Deutschland |3601.02   |2457.34   |
|AUF-10002 |2025-01-01  |Westeuropa|Niederlande |2318.08   |1703.33   |
|AUF-10003 |2025-01-01  |Nordeuropa|Finnland    |7947.83   |6109.5    |
|AUF-10004 |2025-01-01  |DACH      |Deutschland |3039.03   |2095.41   |
|AUF-10005 |2025-01-01  |Südeuropa |Spanien     |2881.56   |1942.31   |
+----------+------------+----------+------------+----------+----------+
only showing top 5 rows


## 3. Silver Layer: Data Cleaning, Enrichment & Quality Assurance
- Bereinigung von führenden/nachlaufenden Whitespaces
- Validierung von Geschäftsregeln (Umsatz > 0, Kosten >= 0)
- Exakte Neuberechnung von `Gewinn_EUR` und `Gewinnmarge_Prozent`
- Deduplizierung über `Bestell_ID`


In [3]:
text_cols = ["Bestell_ID", "Region", "Land", "Stadt", "Kundensegment", "Vertriebskanal", "Produktkategorie", "Produktname"]
df_cleaned = df_bronze
for c in text_cols:
    df_cleaned = df_cleaned.withColumn(c, F.trim(F.col(c)))

# Validierungs-Filter
valid_filter = (
    F.col("Bestell_ID").isNotNull() &
    F.col("Bestelldatum").isNotNull() &
    (F.col("Umsatz_EUR") > 0) &
    (F.col("Kosten_EUR") >= 0)
)

df_silver = (
    df_cleaned
    .filter(valid_filter)
    .dropDuplicates(["Bestell_ID"])
    .withColumn("Jahr", F.year("Bestelldatum"))
    .withColumn("Quartal", F.concat(F.lit("Q"), F.quarter("Bestelldatum")))
    .withColumn("Gewinn_EUR", F.round(F.col("Umsatz_EUR") - F.col("Kosten_EUR"), 2))
    .withColumn("Gewinnmarge_Prozent", F.round((F.col("Gewinn_EUR") / F.col("Umsatz_EUR")) * 100, 2))
    .withColumn("Zielerreichung_Quote", F.round(F.col("Umsatz_EUR") / F.col("Zielumsatz_EUR"), 4))
)

df_silver.createOrReplaceTempView("silver_orders")

total_silver = df_silver.count()
print(f"✨ Silver-Layer validiert: {total_silver:,} bereinigte Datensätze")


✨ Silver-Layer validiert: 4,591 bereinigte Datensätze


### 🛡️ Data Quality Assertion Checks


In [4]:
dq_checks = {
    "1. Primärschlüssel-Eindeutigkeit (Bestell_ID)": df_silver.select("Bestell_ID").distinct().count() == total_silver,
    "2. Keine NULL-Werte in Pflichtfeldern": df_silver.filter(F.col("Region").isNull() | F.col("Umsatz_EUR").isNull()).count() == 0,
    "3. Rabattquote im validen Bereich [0.0 - 1.0]": df_silver.filter(~F.col("Rabatt_Quote").between(0.0, 1.0)).count() == 0,
    "4. Keine negativen Umsätze": df_silver.filter(F.col("Umsatz_EUR") <= 0).count() == 0
}

all_passed = True
for check_name, passed in dq_checks.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status} | {check_name}")
    if not passed:
        all_passed = False

assert all_passed, "Data Quality Gate nicht bestanden!"
print("🎯 Alle 4 Qualitätsprüfungen erfolgreich bestanden.")


✅ PASS | 1. Primärschlüssel-Eindeutigkeit (Bestell_ID)
✅ PASS | 2. Keine NULL-Werte in Pflichtfeldern
✅ PASS | 3. Rabattquote im validen Bereich [0.0 - 1.0]
✅ PASS | 4. Keine negativen Umsätze
🎯 Alle 4 Qualitätsprüfungen erfolgreich bestanden.


## 4. Gold Layer: Executive Sales Marts & KPI-Berechnung
> ⚠️ **Methodischer Hinweis zur Margenberechnung:**  
> Die Gewinnmarge wird im Gold-Layer betriebswirtschaftlich korrekt **volumengewichtet** berechnet:  
> `SUM(Gewinn_EUR) / SUM(Umsatz_EUR)` anstelle eines ungewichteten `AVG(Gewinnmarge_Prozent)`.


In [5]:
df_gold_regional_kpis = spark.sql("""
    SELECT 
        Region,
        COUNT(DISTINCT Bestell_ID) AS Anzahl_Auftraege,
        ROUND(SUM(Umsatz_EUR), 2) AS Gesamtumsatz_EUR,
        ROUND(SUM(Gewinn_EUR), 2) AS Deckungsbeitrag_EUR,
        ROUND(AVG(Gewinnmarge_Prozent), 2) AS Marge_Ungewichtet_Avg_Pct,
        ROUND((SUM(Gewinn_EUR) / SUM(Umsatz_EUR)) * 100, 2) AS Marge_Gewichtet_Pct,
        ROUND(AVG(Gewinnmarge_Prozent) - ((SUM(Gewinn_EUR) / SUM(Umsatz_EUR)) * 100), 2) AS Differenz_Prozentpunkte,
        ROUND((SUM(Umsatz_EUR) / SUM(Zielumsatz_EUR)) * 100, 2) AS Zielerreichung_Pct
    FROM silver_orders
    GROUP BY Region
    ORDER BY Gesamtumsatz_EUR DESC
""")

df_gold_regional_kpis.show(truncate=False)


+----------+----------------+----------------+-------------------+-------------------------+-------------------+-----------------------+--------------------+
|Region    |Anzahl_Auftraege|Gesamtumsatz_EUR|Deckungsbeitrag_EUR|Marge_Ungewichtet_Avg_Pct|Marge_Gewichtet_Pct|Differenz_Prozentpunkte|Zielerreichung_Pct  |
+----------+----------------+----------------+-------------------+-------------------------+-------------------+-----------------------+--------------------+
|DACH      |2211            |18821161.01     |12404618.36        |66.95                    |65.91              |1.03                   |104.12              |
|Westeuropa|1126            |9312257.82      |6202189.44         |67.19                    |66.60              |0.59                   |102.85              |
|Nordeuropa|722             |6207611.45      |4101988.19         |66.91                    |66.08              |0.84                   |101.94              |
|Südeuropa |532             |4575482.72      |306024

### 🔍 Drilldown-Analyse: Region × Kundensegment
Hier überprüfen wir exakt, warum der Effekt in der **DACH-Region** mit **+1,03 Prozentpunkten** am stärksten ausgeprägt ist.
Wir berechnen:
1. Anzahl Aufträge je Segment
2. Gesamtumsatz je Segment
3. Umsatz-Anteil des Segments an der jeweiligen Region
4. Volumengewichtete Marge je Segment


In [6]:
df_region_segment = spark.sql("""
    WITH regional_totals AS (
        SELECT Region, SUM(Umsatz_EUR) AS Reg_Gesamtumsatz_EUR
        FROM silver_orders
        GROUP BY Region
    )
    SELECT 
        s.Region,
        s.Kundensegment,
        COUNT(s.Bestell_ID) AS Anzahl_Auftraege,
        ROUND(SUM(s.Umsatz_EUR), 2) AS Segment_Umsatz_EUR,
        ROUND((SUM(s.Umsatz_EUR) / FIRST(r.Reg_Gesamtumsatz_EUR)) * 100, 2) AS Umsatz_Anteil_Region_Pct,
        ROUND((SUM(s.Gewinn_EUR) / SUM(s.Umsatz_EUR)) * 100, 2) AS Marge_Gewichtet_Pct,
        ROUND(AVG(s.Gewinnmarge_Prozent), 2) AS Marge_Ungewichtet_Pct
    FROM silver_orders s
    JOIN regional_totals r ON s.Region = r.Region
    GROUP BY s.Region, s.Kundensegment
    ORDER BY s.Region, Segment_Umsatz_EUR DESC
""")

df_region_segment.show(15, truncate=False)


+----------+-----------------------+----------------+------------------+-----------------------+-------------------+---------------------+
|Region    |Kundensegment          |Anzahl_Auftraege|Segment_Umsatz_EUR|Umsatz_Anteil_Region_Pct|Marge_Gewichtet_Pct|Marge_Ungewichtet_Pct|
+----------+-----------------------+----------------+------------------+-----------------------+-------------------+---------------------+
|DACH      |Großkunden (Enterprise)|1006            |12240290.33       |65.03                  |65.32              |65.78                |
|DACH      |Mittelstand (KMU)      |833             |5373521.59        |28.55                  |66.57              |67.43                |
|DACH      |Kleingewerbe           |372             |1207349.28        |6.41                   |69.03              |69.01                |
|Nordeuropa|Großkunden (Enterprise)|296             |3677182.55        |59.24                  |65.37              |65.84                |
|Nordeuropa|Mittelstand (K

## 5. Fazit & Business Insights
1. **Marktführerschaft DACH**: Der DACH-Raum repräsentiert über **48 % des europäischen Gesamtumsatzes** (18,8 Mio. €).
2. **Ursache der Margenverzerrung nachgewiesen**:
   - Die DACH-Region weist den **höchsten Enterprise-Umsatzanteil aller Regionen auf (65,03 %)**.
   - Großkunden erzielen die niedrigste Marge (**65,32 %** ggü. 69,03 % bei Kleingewerbe).
   - Weil Großkunden 65 % des Umsatzes, aber nur 45,5 % der Auftragsanzahl ausmachen, gewichtet das ungewichtete `AVG(Marge)` die margenschwächeren Großaufträge zu schwach. Daher fällt das arithmetische Mittel in DACH um **+1,03 Prozentpunkte** zu hoch aus.
3. **Tableau Integration**: Das aggregierte Gold-Modell speist direkt das **Tableau Executive Dashboard**.
